# Análisis de Ventas por Categoría de Producto

## Autor: Agustín Pluda

## Descripción

Este notebook continúa el trabajo de `product_classifier.ipynb`, donde se construyó un modelo de Machine Learning para clasificar los productos del catálogo de Online Retail II en categorías comerciales.

Acá se toma el catálogo ya categorizado (`data/processed/products_categorized.csv`) y se lo cruza contra las transacciones originales para analizar el comportamiento de las ventas según categoría: participación en el revenue, estacionalidad, ticket promedio, mix por país y productos destacados dentro de cada categoría.

## 0. Importación de librerías

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")
%matplotlib inline

## 1. Carga de datos

### 1.1 Transacciones y catálogo categorizado

In [2]:
transacciones = pd.read_csv("data/main/online_retail_sales.csv")
catalogo = pd.read_csv("data/processed/products_categorized.csv")

print(f"Transacciones: {transacciones.shape}")
print(f"Catalogo     : {catalogo.shape}")

catalogo["categoria_final"].value_counts()

Transacciones: (1003417, 12)
Catalogo     : (4724, 14)


categoria_final
Cocina y Mesa                          860
Sin clasificar                         730
Papelería y Tarjetería                 472
Velas e Iluminación                    447
Joyería y Bijouterie                   396
Decoración del Hogar                   267
Navidad                                244
Cajas y Almacenamiento                 238
Bolsos y Carteras                      228
Espejos, Relojes y Arte de Pared       189
Pascua y Fiestas                       155
Textiles del Hogar                     150
Jardín y Exterior                      129
Marcos y Fotografía                     95
Carteles y Señalética                   76
Botellas de Agua Caliente y Confort     48
Name: count, dtype: int64

## 2. Merge con las categorías

### 2.1 Merge por `stock_code` normalizado

In [3]:
transacciones["stock_code"] = transacciones["stock_code"].str.strip().str.upper()

df_con_categoria = transacciones.merge(
    catalogo[["stock_code", "categoria_final"]],
    on="stock_code",
    how="left",
)

print(f"Filas originales : {len(transacciones):,}")
print(f"Filas post-merge  : {len(df_con_categoria):,}")
print(f"Diferencia        : {len(df_con_categoria) - len(transacciones)}  (deberia ser 0)")

print(f"Sin categoria (vouchers u otros no catalogados): {df_con_categoria['categoria_final'].isna().sum()}")
df_con_categoria.loc[df_con_categoria["categoria_final"].isna(), "categoria_final"] = "Sin categorizar (voucher/otro)"

Filas originales : 1,003,417
Filas post-merge  : 1,003,417
Diferencia        : 0  (deberia ser 0)
Sin categoria (vouchers u otros no catalogados): 77


### 2.2 Verificación de cierre de revenue

In [4]:
revenue_por_categoria = df_con_categoria.groupby("categoria_final")["total_price"].sum().sort_values(ascending=False)

print(f"Revenue total (transacciones)     : {transacciones['total_price'].sum():,.2f}")
print(f"Revenue total (suma por categoria): {revenue_por_categoria.sum():,.2f}")

revenue_por_categoria

Revenue total (transacciones)     : 19,645,617.79
Revenue total (suma por categoria): 19,645,617.79


categoria_final
Cocina y Mesa                          4490145.95
Bolsos y Carteras                      2017224.44
Papelería y Tarjetería                 1661688.74
Sin clasificar                         1652964.02
Velas e Iluminación                    1652434.17
Cajas y Almacenamiento                 1478892.06
Decoración del Hogar                   1332728.29
Textiles del Hogar                      951579.37
Navidad                                 924183.34
Pascua y Fiestas                        754055.40
Botellas de Agua Caliente y Confort     692074.23
Jardín y Exterior                       552178.58
Carteles y Señalética                   539207.58
Espejos, Relojes y Arte de Pared        433733.45
Marcos y Fotografía                     376418.14
Joyería y Bijouterie                    134353.86
Sin categorizar (voucher/otro)            1756.17
Name: total_price, dtype: float64